<a href="https://colab.research.google.com/github/mirandaurban/tc3009c/blob/main/Cloud/CCBD_Classwork_05_Curve_Fitting_with_pySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CCBD Classwork 05: Curve Fitting with pySpark

In [ ]:
#All imports
import os

from google.colab import files
from pyspark import SparkContext, SparkConf,SQLContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import isnan, when, count, col, lit
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
#os.environ["SPARK_HOME"] = "/content/spark-2.4.8-bin-hadoop2.7"

Run a local spark session to test your installation:

In [ ]:
# create sparksession and sparkcontext
sc = SparkContext.getOrCreate(SparkConf().setMaster("local[*]"))

spark = SparkSession \
    .builder \
    .getOrCreate()

sqlContext = SQLContext(sc)

/usr/local/lib/python3.13/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [ ]:
# to retrieve SparkContext version
spark.version

'4.0.5'

## Importar Google Drive

In [ ]:
#Mount Google drive, read local file. The file was previously uploaded to Google Drive
from google.colab import drive
drive.mount("/content/drive")
%ls

Mounted at /content/drive
drive/  sample_data/


## Importar Datos

Base de datos Boston Housing

El conjunto de datos de este modelo proviene del repositorio “UCI Machine Learning”. Estos datos se recopilaron en in 1978, y cada una de las 506 entradas representan datos agregados de 14 características de casas en diversos barrios de Boston. La base de datos Boston Housing contiene los datos originales de Harrison y Rubinfeld (1979).

Los datos tienen las siguientes características, siendo medv la variable de objetivo o independiente

crim - Crimen per cápita por ciudad

zn - proporción de terrenos residenciales divididos en zonas para lotes de más de 25,000 pies cuadrados

indus - proporción de acres de negocios no minoristas por ciudad

chas - variable ficticia de Charles River (= 1 si el tramo limita el río, 0 de lo contrario)

nox - concentración de óxidos nítricos (partes por 10 millones)

rm - número promedio de habitaciones por vivienda

age - proporción de unidades ocupadas por sus propietarios construidas antes de 1940

dis - Distancias desproporcionadas a cinco centros de empleo de Boston

rad - índice de accesibilidad a las autopistas radiales

tax - tasa de impuesto a la propiedad de valor completo por USD 10,000

ptratio - colegios por localidad

black - 1000 (B - 0,63)^ 2, donde B es la proporción de negros por ciudad

lstat - porcentaje de estado inferior de la población

medv - valor mediano de las viviendas ocupadas por sus propietarios en USD 1000

In [ ]:
path_csv="/content/drive/MyDrive/Colab Notebooks/cn/BostonHousing.csv"
dataset = spark.read.csv(path_csv,inferSchema=True, header =True)
dataset.show(10)

+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|   crim|  zn|indus|chas|  nox|   rm|  age|   dis|rad|tax|ptratio| black|lstat|medv|
+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|0.00632|18.0| 2.31|   0|0.538|6.575| 65.2|  4.09|  1|296|   15.3| 396.9| 4.98|24.0|
|0.02731| 0.0| 7.07|   0|0.469|6.421| 78.9|4.9671|  2|242|   17.8| 396.9| 9.14|21.6|
|0.02729| 0.0| 7.07|   0|0.469|7.185| 61.1|4.9671|  2|242|   17.8|392.83| 4.03|34.7|
|0.03237| 0.0| 2.18|   0|0.458|6.998| 45.8|6.0622|  3|222|   18.7|394.63| 2.94|33.4|
|0.06905| 0.0| 2.18|   0|0.458|7.147| 54.2|6.0622|  3|222|   18.7| 396.9| 5.33|36.2|
|0.02985| 0.0| 2.18|   0|0.458| 6.43| 58.7|6.0622|  3|222|   18.7|394.12| 5.21|28.7|
|0.08829|12.5| 7.87|   0|0.524|6.012| 66.6|5.5605|  5|311|   15.2| 395.6|12.43|22.9|
|0.14455|12.5| 7.87|   0|0.524|6.172| 96.1|5.9505|  5|311|   15.2| 396.9|19.15|27.1|
|0.21124|12.5| 7.87|   0|0.524|5.631|100.0|6.0821|  5|311|   15.2

## Modelos de Regresion Lineal y No Lineal


Linear Regression model is one the oldest and widely used machine learning approach which assumes a relationship between dependent and independent variables. For example, a modeler might want to predict the forecast of the rain based on the humidity ratio. Linear Regression consists of the best fitting line through the scattered points on the graph and the best fitting line is known as the regression line.

The goal of this exercise to predict the housing prices by the given features. Let's predict the prices of the Boston Housing dataset by considering MEDV as the output variable and all the other variables as input.

Download the dataset from [here](https://github.com/asifahmed90/pyspark-ML-in-Colab/blob/master/BostonHousing.csv) and keep it somewhere on your computer. Load the dataset into your Colab directory from your local system:

For our linear regression model we need to import two modules from Pyspark i.e. Vector Assembler and Linear Regression. Vector Assembler is a transformer that assembles all the features into one vector from multiple columns that contain type double. We could have used StringIndexer if any of our columns contains string values to convert it into numeric values. Luckily, the BostonHousing dataset only contains double values, so we don't need to worry about StringIndexer for now.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

Notice that we used InferSchema inside read.csv mofule. InferSchema enables us to infer automatically different data types for each column.

Let us print look into the dataset to see the data types of each column:

In [ ]:
dataset.printSchema()

root
 |-- crim: double (nullable = true)
 |-- zn: double (nullable = true)
 |-- indus: double (nullable = true)
 |-- chas: integer (nullable = true)
 |-- nox: double (nullable = true)
 |-- rm: double (nullable = true)
 |-- age: double (nullable = true)
 |-- dis: double (nullable = true)
 |-- rad: integer (nullable = true)
 |-- tax: integer (nullable = true)
 |-- ptratio: double (nullable = true)
 |-- black: double (nullable = true)
 |-- lstat: double (nullable = true)
 |-- medv: double (nullable = true)



Next step is to convert all the features from different columns into a single column and let's call this new vector column as 'Attributes' in the outputCol.

In [ ]:
#Input all the features in one vector column
assembler = VectorAssembler(inputCols=['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'black', 'lstat'], outputCol = 'Attributes')

output = assembler.transform(dataset)

#Input vs Output
finalized_data = output.select("Attributes","medv")

finalized_data.show(10)

+--------------------+----+
|          Attributes|medv|
+--------------------+----+
|[0.00632,18.0,2.3...|24.0|
|[0.02731,0.0,7.07...|21.6|
|[0.02729,0.0,7.07...|34.7|
|[0.03237,0.0,2.18...|33.4|
|[0.06905,0.0,2.18...|36.2|
|[0.02985,0.0,2.18...|28.7|
|[0.08829,12.5,7.8...|22.9|
|[0.14455,12.5,7.8...|27.1|
|[0.21124,12.5,7.8...|16.5|
|[0.17004,12.5,7.8...|18.9|
+--------------------+----+
only showing top 10 rows


Here, 'Attributes' are in the input features from all the columns and 'medv' is the target column.
Next, we should split the training and testing data according to our dataset (0.8 and 0.2 in this case).

In [ ]:
#Split training and testing data
train_data,test_data = finalized_data.randomSplit([0.8,0.2])


regressor = LinearRegression(featuresCol = 'Attributes', labelCol = 'medv')

#Learn to fit the model from training set
regressor = regressor.fit(train_data)

#To predict the prices on testing set
#pred_lr = regressor.evaluate(test_data)

#Predict the model
#pred_lr.predictions.show(10)

#It can also be done in this way
pred_lr = regressor.transform(test_data)
pred_lr.select('Attributes', 'medv', 'prediction').show(15)

+--------------------+----+------------------+
|          Attributes|medv|        prediction|
+--------------------+----+------------------+
|[0.01096,55.0,2.2...|22.0| 27.27111583100528|
|[0.01439,60.0,2.9...|29.1|31.792605871127186|
|[0.01778,95.0,1.4...|32.9|30.752353623736948|
|[0.0187,85.0,4.15...|23.1|25.394414522128333|
|[0.02187,60.0,2.9...|31.1|32.311693057088895|
|[0.02729,0.0,7.07...|34.7|  30.1970259299773|
|[0.02731,0.0,7.07...|21.6|24.670324231675508|
|[0.02899,40.0,1.2...|26.6| 21.50526102085373|
|[0.03578,20.0,3.3...|45.4| 38.69103895520703|
|[0.03615,80.0,4.9...|27.9| 32.37114309075773|
|[0.03768,80.0,1.5...|34.6|  34.8360956028221|
|[0.03961,0.0,5.19...|21.1|20.444150921625994|
|[0.04011,80.0,1.5...|33.3| 36.31610760730769|
|[0.04113,25.0,4.8...|28.0|28.237716156370944|
|[0.05023,35.0,6.0...|17.1| 19.88800418402558|
+--------------------+----+------------------+
only showing top 15 rows


In [ ]:
import numpy as np
print ("Note: the last rows are the information for Intercept")
print ("##","-------------------------------------------------")
print ("##","  Estimate   |   Std.Error | t Values  |  P-value")
coef = np.append(list(regressor.coefficients),regressor.intercept)
Summary=regressor.summary

for i in range(len(Summary.pValues)):
    print ("##",'{:10.6f}'.format(coef[i]),\
    '{:10.6f}'.format(Summary.coefficientStandardErrors[i]),\
    '{:8.3f}'.format(Summary.tValues[i]),\
    '{:10.6f}'.format(Summary.pValues[i]))

print ("##",'---')
print ("##","Mean squared error: % .6f" \
        % Summary.meanSquaredError, ", \
        RMSE: % .6f" \
        % Summary.rootMeanSquaredError )
print ("##","Multiple R-squared: %f" % Summary.r2, ", \
        Total iterations: %i"% Summary.totalIterations)

Note: the last rows are the information for Intercept
## -------------------------------------------------
##   Estimate   |   Std.Error | t Values  |  P-value
##  -0.107845   0.033935   -3.178   0.001600
##   0.057160   0.015446    3.701   0.000246
##   0.031600   0.068771    0.459   0.646133
##   1.835842   0.971163    1.890   0.059440
## -16.052164   4.235516   -3.790   0.000174
##   3.598897   0.457405    7.868   0.000000
##  -0.001803   0.015033   -0.120   0.904581
##  -1.576458   0.229127   -6.880   0.000000
##   0.332475   0.073258    4.538   0.000008
##  -0.014061   0.004164   -3.377   0.000807
##  -0.994064   0.150365   -6.611   0.000000
##   0.009140   0.002926    3.124   0.001915
##  -0.544471   0.057038   -9.546   0.000000
##  38.623407   5.617118    6.876   0.000000
## ---
## Mean squared error:  22.272534 ,         RMSE:  4.719379
## Multiple R-squared: 0.739938 ,         Total iterations: 0


We can also print the coefficient and intercept of the regression model by using the following command:

In [ ]:
#coefficient of the regression model
#coeff = regressor.coefficients

#X and Y intercept
#intr = regressor.intercept

print ("The coefficient of the model is : ", regressor.coefficients)
print ("The Intercept of the model is : ", regressor.intercept)


The coefficient of the model is :  [-0.10784458449017964,0.05716003046908954,0.03159976808880348,1.83584245757059,-16.052163581837437,3.5988972012242315,-0.0018031949325578978,-1.576458458889696,0.3324747660860384,-0.014061202231793444,-0.9940637650857713,0.00914022895424654,-0.5444708300820041]
The Intercept of the model is :  38.62340673049862


In [ ]:
#Carry out random forrest regression
from pyspark.ml.regression import RandomForestRegressor
rfr = RandomForestRegressor(featuresCol = 'Attributes', labelCol = 'medv')

#Learn to fit the model from training set
rfr = rfr.fit(train_data)

#To predict the prices on testing set
pred_rfr = rfr.transform(test_data)

# Select example rows to display.
pred_rfr.select('Attributes', 'medv', 'prediction').show(10)

+--------------------+----+------------------+
|          Attributes|medv|        prediction|
+--------------------+----+------------------+
|[0.01096,55.0,2.2...|22.0|23.030880424293922|
|[0.01439,60.0,2.9...|29.1|28.761926780315044|
|[0.01778,95.0,1.4...|32.9|  33.8960228072652|
|[0.0187,85.0,4.15...|23.1|  25.3721564785857|
|[0.02187,60.0,2.9...|31.1| 30.74009124160137|
|[0.02729,0.0,7.07...|34.7| 36.15855247113997|
|[0.02731,0.0,7.07...|21.6|22.502458786185105|
|[0.02899,40.0,1.2...|26.6|25.770332154891285|
|[0.03578,20.0,3.3...|45.4| 45.34538293650794|
|[0.03615,80.0,4.9...|27.9| 28.94391487555314|
+--------------------+----+------------------+
only showing top 10 rows


In [ ]:
#Carry out Gradient-boosted tree regression
from pyspark.ml.regression import GBTRegressor
gbt = GBTRegressor(featuresCol = 'Attributes', labelCol = 'medv')

#Learn to fit the model from training set
gbt = gbt.fit(train_data)

#To predict the prices on testing set
pred_gbt = gbt.transform(test_data)

# Select example rows to display.
pred_gbt.select('Attributes', 'medv', 'prediction').show(10)

+--------------------+----+------------------+
|          Attributes|medv|        prediction|
+--------------------+----+------------------+
|[0.01096,55.0,2.2...|22.0| 23.00973513594428|
|[0.01439,60.0,2.9...|29.1| 31.43635826581452|
|[0.01778,95.0,1.4...|32.9|31.745013172320217|
|[0.0187,85.0,4.15...|23.1|23.596765491761982|
|[0.02187,60.0,2.9...|31.1|32.617788367990734|
|[0.02729,0.0,7.07...|34.7|30.925632905122388|
|[0.02731,0.0,7.07...|21.6|19.167843889414975|
|[0.02899,40.0,1.2...|26.6|22.798144639576364|
|[0.03578,20.0,3.3...|45.4| 47.13244038714953|
|[0.03615,80.0,4.9...|27.9|  30.8554377283758|
+--------------------+----+------------------+
only showing top 10 rows


## Analisis Estadistico de la Regresion

Once we are done with the basic linear regression operation, we can go a bit further and analyze our model statistically by importing RegressionEvaluator module from Pyspark.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
#Evaluado la regresion lineal
eval_lr = RegressionEvaluator(labelCol="medv", predictionCol="prediction", metricName="rmse")

print("Linear regression model\n")

# Root Mean Square Error
rmse = eval_lr.evaluate(pred_lr)
#rmse = eval_lr.evaluate(pred_lr.predictions)
print("RMSE: %.3f" % rmse)

# Mean Square Error
#mse = eval_lr.evaluate(pred_lr.predictions, {eval_lr.metricName: "mse"})
#print("MSE: %.3f" % mse)

# Mean Absolute Error
#mae = eval_lr.evaluate(pred_lr.predictions, {eval_lr.metricName: "mae"})
#print("MAE: %.3f" % mae)

# r2 - coefficient of determination
#r2 = eval_lr.evaluate(pred_lr.predictions, {eval_lr.metricName: "r2"})
#print("r2: %.3f" %r2)

# r2 - coefficient of determination
r2 = eval_lr.evaluate(pred_lr, {eval_lr.metricName: "r2"})
print("r2: %.3f" %r2)

Linear regression model

RMSE: 4.577
r2: 0.727


In [ ]:
eval_rfr = RegressionEvaluator(labelCol="medv", predictionCol="prediction", metricName="rmse")
#Evaluando el RFR

print("Regression Forrest model\n")

# Root Mean Square Error
#rmse = eval_rfr.evaluate(pred_rfr.predictions)
rmse = eval_rfr.evaluate(pred_rfr)

print("RMSE: %.3f" % rmse)

# Mean Square Error
mse = eval_rfr.evaluate(pred_rfr, {eval_rfr.metricName: "mse"})
print("MSE: %.3f" % mse)

# Mean Absolute Error
mae = eval_rfr.evaluate(pred_rfr, {eval_rfr.metricName: "mae"})
print("MAE: %.3f" % mae)

# r2 - coefficient of determination
r2 = eval_rfr.evaluate(pred_rfr, {eval_rfr.metricName: "r2"})
print("r2: %.3f" %r2)

Regression Forrest model

RMSE: 3.419
MSE: 11.691
MAE: 2.508
r2: 0.878


In [ ]:
eval_gbt = RegressionEvaluator(labelCol="medv", predictionCol="prediction", metricName="rmse")
#Evaluando GBTR

print("Gradient Boot Tree model\n")

# Root Mean Square Error
#rmse = eval_rfr.evaluate(pred_rfr.predictions)
rmse = eval_gbt.evaluate(pred_gbt)

print("RMSE: %.3f" % rmse)

# Mean Square Error
mse = eval_gbt.evaluate(pred_gbt, {eval_gbt.metricName: "mse"})
print("MSE: %.3f" % mse)

# Mean Absolute Error
mae = eval_gbt.evaluate(pred_gbt, {eval_gbt.metricName: "mae"})
print("MAE: %.3f" % mae)

# r2 - coefficient of determination
r2 = eval_gbt.evaluate(pred_gbt, {eval_gbt.metricName: "r2"})
print("r2: %.3f" %r2)

Gradient Boot Tree model

RMSE: 3.589
MSE: 12.878
MAE: 2.664
r2: 0.832


# Curve Fitting with Python

## Dataset *BostonH.csv*

Activities to do:
- Consider crim as the target variable and lstat as the independent variable

- Fit the model increasing the order of the independent variable until get the best fit.

- Try linear regression, random forests and gradient boost.


In [ ]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml import Pipeline

# Re-read the dataset (already loaded earlier as `dataset`)
dataset_ex_1 = dataset.select("lstat", "crim").dropna()

train_data, test_data = dataset_ex_1.randomSplit([0.8, 0.2], seed=42)

evaluator = RegressionEvaluator(
    labelCol="crim", predictionCol="prediction", metricName="rmse"
)

def evaluate_model(pred_df, name):
    rmse = evaluator.evaluate(pred_df)
    mse  = evaluator.evaluate(pred_df, {evaluator.metricName: "mse"})
    mae  = evaluator.evaluate(pred_df, {evaluator.metricName: "mae"})
    r2   = evaluator.evaluate(pred_df, {evaluator.metricName: "r2"})
    return {"model": name, "RMSE": rmse, "MSE": mse, "MAE": mae, "R2": r2}

results = []

In [ ]:
print("Linear Regression: polynomial degree sweep")
for degree in range(1, 6):
    assembler = VectorAssembler(inputCols=["lstat"], outputCol="raw_features")
    poly      = PolynomialExpansion(degree=degree, inputCol="raw_features", outputCol="features")
    lr        = LinearRegression(featuresCol="features", labelCol="crim")

    pipeline = Pipeline(stages=[assembler, poly, lr])
    model    = pipeline.fit(train_data)
    preds    = model.transform(test_data)

    res = evaluate_model(preds, f"LinearReg (deg={degree})")
    results.append(res)
    print(f"degree={degree}: RMSE={res['RMSE']:.4f}  R2={res['R2']:.4f}")

Linear Regression: polynomial degree sweep
degree=1: RMSE=5.3158  R2=0.4645
degree=2: RMSE=5.1346  R2=0.5004
degree=3: RMSE=5.6501  R2=0.3951
degree=4: RMSE=5.4692  R2=0.4332
degree=5: RMSE=5.2888  R2=0.4700


In [ ]:
print("\nRandom Forest: polynomial degree sweep")
for degree in range(1, 6):
    assembler = VectorAssembler(inputCols=["lstat"], outputCol="raw_features")
    poly      = PolynomialExpansion(degree=degree, inputCol="raw_features", outputCol="features")
    rf        = RandomForestRegressor(
        featuresCol="features", labelCol="crim",
        numTrees=100, maxDepth=5, seed=42
    )

    pipeline = Pipeline(stages=[assembler, poly, rf])
    model    = pipeline.fit(train_data)
    preds    = model.transform(test_data)

    res = evaluate_model(preds, f"RandomForest (deg={degree})")
    results.append(res)
    print(f"degree={degree}: RMSE={res['RMSE']:.4f}  R2={res['R2']:.4f}")


Random Forest: polynomial degree sweep
degree=1: RMSE=5.9024  R2=0.3398
degree=2: RMSE=5.9025  R2=0.3398
degree=3: RMSE=5.9025  R2=0.3398
degree=4: RMSE=5.9025  R2=0.3398
degree=5: RMSE=5.9025  R2=0.3398


In [ ]:
print("GBT: polynomial degree sweep")
for degree in range(1, 6):
    assembler = VectorAssembler(inputCols=["lstat"], outputCol="raw_features")
    poly      = PolynomialExpansion(degree=degree, inputCol="raw_features", outputCol="features")
    gbt       = GBTRegressor(
        featuresCol="features", labelCol="crim",
        maxIter=100, maxDepth=5, seed=42
    )

    pipeline = Pipeline(stages=[assembler, poly, gbt])
    model    = pipeline.fit(train_data)
    preds    = model.transform(test_data)

    res = evaluate_model(preds, f"GBT (deg={degree})")
    results.append(res)
    print(f"degree={degree}: RMSE={res['RMSE']:.4f}  R2={res['R2']:.4f}")

GBT: polynomial degree sweep
degree=1: RMSE=5.8549  R2=0.3504
degree=2: RMSE=5.8549  R2=0.3504
degree=3: RMSE=5.8549  R2=0.3504
degree=4: RMSE=5.8549  R2=0.3504
degree=5: RMSE=5.8549  R2=0.3504


In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
print("Ranked results (best fit on top)")
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

best = results_df.iloc[0]
print(f"\nBest configuration: {best['model']}  →  RMSE={best['RMSE']:.4f}, R2={best['R2']:.4f}")

Ranked results (best fit on top)
               model   RMSE     MSE    MAE     R2
   LinearReg (deg=2) 5.1346 26.3639 3.0988 0.5004
   LinearReg (deg=5) 5.2888 27.9710 2.7762 0.4700
   LinearReg (deg=1) 5.3158 28.2575 3.2479 0.4645
   LinearReg (deg=4) 5.4692 29.9117 2.8874 0.4332
   LinearReg (deg=3) 5.6501 31.9232 2.9001 0.3951
         GBT (deg=4) 5.8549 34.2795 3.1753 0.3504
         GBT (deg=1) 5.8549 34.2795 3.1753 0.3504
         GBT (deg=5) 5.8549 34.2795 3.1753 0.3504
         GBT (deg=2) 5.8549 34.2795 3.1753 0.3504
         GBT (deg=3) 5.8549 34.2795 3.1753 0.3504
RandomForest (deg=1) 5.9024 34.8387 3.1998 0.3398
RandomForest (deg=2) 5.9025 34.8389 3.2000 0.3398
RandomForest (deg=3) 5.9025 34.8390 3.2001 0.3398
RandomForest (deg=4) 5.9025 34.8390 3.2001 0.3398
RandomForest (deg=5) 5.9025 34.8391 3.2002 0.3398

Best configuration: LinearReg (deg=2)  →  RMSE=5.1346, R2=0.5004


**Conclusión:** El mejor modelo fue la Regresión Lineal de grado 2, con el error más bajo (RMSE = 5.13) y la mejor explicación de los datos (R² = 0.50). Le ganó a Gradient Boost y a Random Forest porque la relación entre lstat y crim no es una línea recta. Los grados más altos (3, 4, 5) se "aprenden de memoria" los datos de entrenamiento y fallan al predecir casos nuevos, y los modelos de árboles, al tener solo una variable de entrada, no tienen con qué lucirse.



## Dataset *fat.csv*

Activities to do:

- Consider Weight is the response variable and chose the independent variable.
- Fit the model increasing the order of the independent variable until get the best fit.
- Try linear regression, random forests and gradient boost.

In [ ]:
path_csv="/content/drive/MyDrive/Colab Notebooks/cn/fat.csv"
dataset_ex_2 = spark.read.csv(path_csv,inferSchema=True, header =True)
dataset_ex_2.show(10)
print(f"\nSchema del dataset:")
dataset_ex_2.printSchema()

+---+------+----+-------+---+------+------+------+-----+----+-----+-----+-----+-----+----+-----+------+-------+-----+
|_c0|brozek|siri|density|age|weight|height|adipos| free|neck|chest|abdom|  hip|thigh|knee|ankle|biceps|forearm|wrist|
+---+------+----+-------+---+------+------+------+-----+----+-----+-----+-----+-----+----+-----+------+-------+-----+
|  1|  12.6|12.3| 1.0708| 23|154.25| 67.75|  23.7|134.9|36.2| 93.1| 85.2| 94.5| 59.0|37.3| 21.9|  32.0|   27.4| 17.1|
|  2|   6.9| 6.1| 1.0853| 22|173.25| 72.25|  23.4|161.3|38.5| 93.6| 83.0| 98.7| 58.7|37.3| 23.4|  30.5|   28.9| 18.2|
|  3|  24.6|25.3| 1.0414| 22| 154.0| 66.25|  24.7|116.0|34.0| 95.8| 87.9| 99.2| 59.6|38.9| 24.0|  28.8|   25.2| 16.6|
|  4|  10.9|10.4| 1.0751| 26|184.75| 72.25|  24.9|164.7|37.4|101.8| 86.4|101.2| 60.1|37.3| 22.8|  32.4|   29.4| 18.2|
|  5|  27.8|28.7|  1.034| 24|184.25| 71.25|  25.6|133.1|34.4| 97.3|100.0|101.9| 63.2|42.2| 24.0|  32.2|   27.7| 17.7|
|  6|  20.6|20.9| 1.0502| 24|210.25| 74.75|  26.5|167.0|

In [ ]:
TARGET = "weight"      # variable objetivo
FEATURE = "age"      # variable independiente

dataset_ex_2.select(FEATURE, TARGET).dropna()

train_data, test_data = dataset_ex_2.randomSplit([0.8, 0.2], seed=42)

evaluator = RegressionEvaluator(
    labelCol=TARGET, predictionCol="prediction", metricName="rmse"
)

def evaluate_model(pred_df, name):
    rmse = evaluator.evaluate(pred_df)
    mse  = evaluator.evaluate(pred_df, {evaluator.metricName: "mse"})
    mae  = evaluator.evaluate(pred_df, {evaluator.metricName: "mae"})
    r2   = evaluator.evaluate(pred_df, {evaluator.metricName: "r2"})
    return {"model": name, "RMSE": rmse, "MSE": mse, "MAE": mae, "R2": r2}
finalized_data.show(10)

+--------------------+----+
|          Attributes|medv|
+--------------------+----+
|[0.00632,18.0,2.3...|24.0|
|[0.02731,0.0,7.07...|21.6|
|[0.02729,0.0,7.07...|34.7|
|[0.03237,0.0,2.18...|33.4|
|[0.06905,0.0,2.18...|36.2|
|[0.02985,0.0,2.18...|28.7|
|[0.08829,12.5,7.8...|22.9|
|[0.14455,12.5,7.8...|27.1|
|[0.21124,12.5,7.8...|16.5|
|[0.17004,12.5,7.8...|18.9|
+--------------------+----+
only showing top 10 rows


In [ ]:
print("Linear Regression: polynomial degree sweep")
for degree in range(1, 6):
    pipeline = Pipeline(stages=[
        VectorAssembler(inputCols=[FEATURE], outputCol="raw_features"),
        PolynomialExpansion(degree=degree, inputCol="raw_features", outputCol="features"),
        LinearRegression(featuresCol="features", labelCol=TARGET),
    ])
    model = pipeline.fit(train_data)
    preds = model.transform(test_data)
    res = evaluate_model(preds, f"LinearReg (deg={degree})")
    results.append(res)
    print(f"degree={degree}: RMSE={res['RMSE']:.4f}  R2={res['R2']:.4f}")


Linear Regression: polynomial degree sweep
degree=1: RMSE=28.5358  R2=-0.0476
degree=2: RMSE=28.5266  R2=-0.0469
degree=3: RMSE=28.5295  R2=-0.0471
degree=4: RMSE=28.5766  R2=-0.0506
degree=5: RMSE=29.1331  R2=-0.0919


In [ ]:
print("Random Forest: polynomial degree sweep")
for degree in range(1, 6):
    pipeline = Pipeline(stages=[
        VectorAssembler(inputCols=[FEATURE], outputCol="raw_features"),
        PolynomialExpansion(degree=degree, inputCol="raw_features", outputCol="features"),
        RandomForestRegressor(featuresCol="features", labelCol=TARGET,
                              numTrees=100, maxDepth=5, seed=42),
    ])
    model = pipeline.fit(train_data)
    preds = model.transform(test_data)
    res = evaluate_model(preds, f"RandomForest (deg={degree})")
    results.append(res)
    print(f"degree={degree}: RMSE={res['RMSE']:.4f}  R2={res['R2']:.4f}")

Random Forest: polynomial degree sweep
degree=1: RMSE=30.3136  R2=-0.1822
degree=2: RMSE=30.3136  R2=-0.1822
degree=3: RMSE=30.3136  R2=-0.1822
degree=4: RMSE=30.3136  R2=-0.1822
degree=5: RMSE=30.3136  R2=-0.1822


In [ ]:
print("GBT: degree sweep")
for degree in range(1, 6):
    pipeline = Pipeline(stages=[
        VectorAssembler(inputCols=[FEATURE], outputCol="raw_features"),
        PolynomialExpansion(degree=degree, inputCol="raw_features", outputCol="features"),
        GBTRegressor(featuresCol="features", labelCol=TARGET,
                     maxIter=100, maxDepth=5, seed=42),
    ])
    model = pipeline.fit(train_data)
    preds = model.transform(test_data)
    res = evaluate_model(preds, f"GBT (deg={degree})")
    results.append(res)
    print(f"degree={degree}: RMSE={res['RMSE']:.4f}  R2={res['R2']:.4f}")

GBT: degree sweep
degree=1: RMSE=32.1967  R2=-0.3336
degree=2: RMSE=32.1967  R2=-0.3336
degree=3: RMSE=32.1967  R2=-0.3336
degree=4: RMSE=32.1967  R2=-0.3336
degree=5: RMSE=32.1967  R2=-0.3336


In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
print("Ranked results (best fit on top)")
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

best = results_df.iloc[0]
print(f"\nBest configuration: {best['model']}  →  RMSE={best['RMSE']:.4f}, R2={best['R2']:.4f}")

Ranked results (best fit on top)
               model    RMSE       MSE     MAE      R2
   LinearReg (deg=2)  5.1346   26.3639  3.0988  0.5004
   LinearReg (deg=5)  5.2888   27.9710  2.7762  0.4700
   LinearReg (deg=1)  5.3158   28.2575  3.2479  0.4645
   LinearReg (deg=4)  5.4692   29.9117  2.8874  0.4332
   LinearReg (deg=3)  5.6501   31.9232  2.9001  0.3951
         GBT (deg=4)  5.8549   34.2795  3.1753  0.3504
         GBT (deg=2)  5.8549   34.2795  3.1753  0.3504
         GBT (deg=3)  5.8549   34.2795  3.1753  0.3504
         GBT (deg=1)  5.8549   34.2795  3.1753  0.3504
         GBT (deg=5)  5.8549   34.2795  3.1753  0.3504
RandomForest (deg=1)  5.9024   34.8387  3.1998  0.3398
RandomForest (deg=2)  5.9025   34.8389  3.2000  0.3398
RandomForest (deg=4)  5.9025   34.8390  3.2001  0.3398
RandomForest (deg=3)  5.9025   34.8390  3.2001  0.3398
RandomForest (deg=5)  5.9025   34.8391  3.2002  0.3398
   LinearReg (deg=2) 28.5266  813.7641 23.4584 -0.0469
   LinearReg (deg=3) 28.5295  81

**Conclusión**: En la parte donde los resultados tienen sentido, vuelve a ganar la Regresión Lineal de grado 2 (RMSE = 5.13, R² = 0.50), lo que confirma que la relación es curva y que un modelo simple y bien ajustado le gana a los modelos más complejos cuando solo hay una variable. En la parte de abajo todos los modelos fallan (R² negativo), lo que significa que ahí no había una relación real que aprender; aun así, la Regresión Lineal de grado 2 fue la que menos se equivocó, así que también es la más confiable cuando las cosas salen mal.